# GRPO: Group Relative Policy Optimization

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/grpo-objective)

We implement the GRPO advantage computation and clipped surrogate objective from scratch on a toy group of sampled completions, then visualize how group-relative normalization turns raw 0/1 rewards into a signed learning signal.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Group-relative advantage

For a group of $G$ sampled completions with rewards $r_1,\dots,r_G$:

$$\hat A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r) + \epsilon}$$

In [ ]:
def group_relative_advantage(rewards, eps=1e-8):
    rewards = np.asarray(rewards, dtype=float)
    mean, std = rewards.mean(), rewards.std()
    return (rewards - mean) / (std + eps)

# Four completions for one math prompt: verifier reward is 1 (correct) or 0 (incorrect)
rewards = np.array([1.0, 0.0, 1.0, 0.0])
advantages = group_relative_advantage(rewards)
for i, (r, a) in enumerate(zip(rewards, advantages), 1):
    print(f"completion {i}: reward={r:.1f}  advantage={a:+.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#34d399' if r > 0 else '#f43f5e' for r in rewards]
axes[0].bar(range(1, 5), rewards, color=colors)
axes[0].set_title('Raw verifier reward'); axes[0].set_xlabel('completion'); axes[0].set_ylim(-1.5, 1.5)
axes[1].bar(range(1, 5), advantages, color=colors)
axes[1].axhline(0, color='white', lw=0.5)
axes[1].set_title('Group-relative advantage'); axes[1].set_xlabel('completion'); axes[1].set_ylim(-1.5, 1.5)
plt.tight_layout(); plt.show()
print("Group normalization turns flat 0/1 rewards into a signed push: up for correct, down for incorrect.")

## 2 — A harder prompt: all completions score the same

If every completion in the group gets the *same* reward, `std` is 0 and there is no learning signal — exactly the right behavior, since there is nothing to prefer between identical outcomes. The `eps` in the denominator only prevents a division-by-zero crash; it does not fabricate a signal.

In [ ]:
uniform_rewards = np.array([1.0, 1.0, 1.0, 1.0])
print('advantages when all completions agree:', group_relative_advantage(uniform_rewards))

## 3 — Clipped surrogate objective (per-token, PPO-style)

$$J(\theta) = \frac{1}{G}\sum_i \frac{1}{|o_i|}\sum_t \min\big(r_{i,t}\hat A_i,\ \text{clip}(r_{i,t}, 1-\epsilon, 1+\epsilon)\hat A_i\big)$$

We simulate token-level probability ratios $r_{i,t} = \pi_\theta / \pi_{\theta_{old}}$ directly (skipping an actual language model) to isolate how clipping behaves.

In [ ]:
def clipped_objective(ratios, advantage, clip_eps=0.2):
    unclipped = ratios * advantage
    clipped = np.clip(ratios, 1 - clip_eps, 1 + clip_eps) * advantage
    return np.minimum(unclipped, clipped)

ratios = np.linspace(0.4, 1.8, 200)
for adv, color, label in [(1.0, '#34d399', 'advantage = +1 (good completion)'),
                           (-1.0, '#f43f5e', 'advantage = -1 (bad completion)')]:
    obj = clipped_objective(ratios, adv)
    plt.plot(ratios, obj, color=color, label=label)
plt.axvline(1.0, color='white', lw=0.5, ls='--')
plt.xlabel('probability ratio r_t = pi_theta / pi_theta_old'); plt.ylabel('per-token objective')
plt.title('PPO-style clipping used inside GRPO'); plt.legend(); plt.tight_layout(); plt.show()
print("For a good completion the objective flattens once r_t > 1+eps: no reward for pushing the ratio further.")
print("For a bad completion it flattens once r_t < 1-eps: no extra penalty for already having suppressed it.")

## ✏️ Your turn

**Task — KL penalty term:** GRPO's full loss subtracts a KL penalty against a frozen reference policy:

$$\mathcal{L} = -J(\theta) + \beta \, D_{KL}(\pi_\theta \,\|\, \pi_{ref})$$

For discrete distributions, $D_{KL}(P\|Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}$. Implement `kl_penalty(p, q)` for two probability vectors over the same token vocabulary, then check that it is exactly `0` when `p == q`, and positive otherwise.

In [ ]:
def kl_penalty(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    # TODO(you): return the KL divergence D_KL(p || q) = sum(p * log(p/q))
    return ...

policy = np.array([0.7, 0.2, 0.1])
reference = np.array([0.5, 0.3, 0.2])
same = np.array([0.7, 0.2, 0.1])

result = kl_penalty(policy, reference)
if result is not None:
    print('KL(policy || reference) =', result)
    print('KL(policy || policy)   =', kl_penalty(policy, same))
    assert kl_penalty(policy, same) < 1e-9
    assert result > 0
    print('Looks right: zero when distributions match, positive otherwise.')

<details><summary>Solution</summary>

```python
def kl_penalty(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(np.sum(p * np.log((p + eps) / (q + eps))))
```
The `eps` inside the log avoids `log(0)` if either distribution has a zero-probability token; it doesn't change the result when `p == q` since the ratio is still 1 for every nonzero entry.
</details>